<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_DIR = '/content/FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git

os.chdir(REPO_DIR)

!python scripts/01_prepare_features.py

import sys
sys.path.append('scripts')

from ml_utils import MODEL_NUMERIC_FEATURES

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Paper findings

**Finding A — Growth Prediction (Part IV):** 90% accuracy on new pages from known brands,
75% on entirely unseen brands — a 15-point drop.

**Methodology questions:**
- What defines 'known brand' in the split, and could pages from the same brand appear in both train and test? (even different pages), and if so, does the model learn brand-level shortcuts
  (a brand's typical publishing cadence, niche, or baseline health score) rather than
  page-level growth signal? This is the same client-leakage question notebook 02 raised
  about `client_hash_id`.
- The report says this was "tested 20 different ways across both methods" with accuracy
  ranging 64%–85% on unseen brands — that's a wide range. What does the *distribution* look
  like, not just the average? A model that's 85% on some brand splits and 64% on others is
  a very different honesty story than a model that's consistently ~75%.
- Top predictor is "Days Visible" (0.16) — is that measured in a window that could overlap
  the label window (a page counted as "visible" during the same days used to judge whether
  it grew)? The paper doesn't show the feature-label timing explicitly.

**Finding B — Zombie Recovery (Part IV):** 99% same-brand vs 97% unseen-brand — only a
2-point gap, much tighter than Finding A's 15-point gap.

**Methodology questions:**
- Why does this model generalize to new brands so much better than Growth Prediction?
  One honest hypothesis: recovery may be driven by broadly transferable signals (content
  age, prior impressions) rather than brand-specific patterns — worth checking whether
  the top features here (Content Age, Impressions) are less brand-coupled than Growth
  Prediction's top feature (Days Visible).
- 59% of zero-traffic pages "came back on their own" — is "recovery" measured over a fixed
  window after the zero-traffic point, and is that window long enough that some "recoveries"
  are actually just noisy sparse-traffic pages crossing the zero threshold randomly rather
  than a real signal-driven comeback?
- The report is written by the company selling the automated fix for exactly this problem
  (zombie recovery). That's not a reason to dismiss the number, but it's a reason to check
  whether the reported metric (accuracy) is the most honest one — precision/recall on the
  minority "does NOT recover" class matters more for a triage tool than raw accuracy, and
  isn't reported here.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
feature_cols = ['impressions_prev_30d', 'avg_position', 'days_since_last_update',
                'log_impressions_90d', 'has_clicks', 'measurable_opportunity']
model_data = df.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining_label']

# BEFORE: random split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xr_tr, yr_tr)
random_acc = rf_random.score(Xr_te, yr_te)

# AFTER: grouped split by client_id (Week 5's design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_id']))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xg_tr, yg_tr)
grouped_acc = rf_grouped.score(Xg_te, yg_te)

print(f'random split accuracy:  {random_acc:.3f}')
print(f'grouped split accuracy: {grouped_acc:.3f}')
print(f'base rate: {max(y.mean(), 1-y.mean()):.3f}')

random split accuracy:  0.717
grouped split accuracy: 0.672
base rate: 0.542


### Random split vs client-grouped split

The random split achieved **71.7% accuracy**, while the client-grouped split achieved **67.2%**. This is a **4.5 percentage-point drop** when the validation split prevents the same client from appearing in both training and validation.

I therefore treat the client-grouped result as the more conservative estimate of generalization for this dataset.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# Week 3 flagged log_impressions_90d (and siblings) as overlapping-window leakage.
# Week 5 included log_impressions_90d and measurable_opportunity.
# Re-run the WITH/WITHOUT test using the same client-grouped validation.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

# Keep the same features used in the Week 5 model
feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update',
    'log_impressions_90d',
    'has_clicks',
    'measurable_opportunity'
]

target_col = 'is_declining_label'

# Build X, y, and client groups
X = model_data[feature_cols].copy()
y = model_data[target_col].astype(int)
groups = model_data['client_id']

# Remove the features flagged as possible leakage
suspects = [
    'log_impressions_90d',
    'measurable_opportunity'
]

X_without_leaky = X.drop(columns=suspects)

# Use client-grouped 5-fold validation
gkf = GroupKFold(n_splits=5)

rf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

score_with = cross_val_score(
    rf,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring='accuracy'
).mean()

score_without = cross_val_score(
    rf,
    X_without_leaky,
    y,
    groups=groups,
    cv=gkf,
    scoring='accuracy'
).mean()

base_rate = max(y.mean(), 1 - y.mean())

print(f'Grouped CV accuracy WITH suspect features:    {score_with:.3f}')
print(f'Grouped CV accuracy WITHOUT suspect features: {score_without:.3f}')
print(f'Base rate: {base_rate:.3f}')

Grouped CV accuracy WITH suspect features:    0.684
Grouped CV accuracy WITHOUT suspect features: 0.635
Base rate: 0.542


### Leakage audit result

The client-grouped validation gives an accuracy of **68.4%** with the suspect features included and **63.5%** after removing them. The majority-class base rate is **54.2%**.

The 4.9 percentage-point difference shows that `log_impressions_90d` and `measurable_opportunity` contribute substantial predictive signal. However, this experiment alone does not prove that the features are leaked. Their final use depends on whether the underlying data is available before the prediction cutoff.

Because the project is intended to predict future decline, the final model should only use features that are available before the prediction window.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim** (from my Week-5 Section 4 write-up): "[paste your actual boldest sentence
here — e.g. 'the model achieves strong precision at identifying declining content']"

**Rewritten, matched to the evidence:**
"On a client-held-out split with leakage-flagged features removed, the model reaches [X]%
accuracy against a [Y]% base rate — a real but modest improvement over guessing the majority
class, not a strong or reliable predictive signal. This is decision-support for prioritizing
manual review, not a claim about generalizing to a client or time period outside this dataset,
and the honest number here is materially lower than what an unaudited random-split run would
have reported."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.